In [1]:
import os
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from typing_extensions import TypedDict
from typing import Literal
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END

In [2]:
load_dotenv()

True

In [3]:
llm = ChatOpenAI(
    api_key=os.environ.get("OPENAI_API_KEY"),
    model="gpt-4o-mini"
)

In [13]:
def ask(query: str):
    response = llm.invoke([
        SystemMessage("""You are a supportive query classifier, 
            classify incoming user query as 'technical' or 'billing' or 'account' or 'product' or 'pricing' or 'other'. 
            Other if its not matching with any of the other categories."""),
        HumanMessage(query)
    ])
    return response.content.lower().strip()

In [10]:
class TicketState(TypedDict):
    query: str
    normalized: str
    classification: Literal["technical", "billing", "account", "product", "pricing", "other"]

In [6]:
def normalize(state: TicketState) -> dict:
    return {
        "normalized": state["query"].lower().strip()
    }

In [7]:
def classify(state: TicketState) -> dict:
    response = ask(state["normalized"])
    return {
        "classification": response 
    }

In [8]:
graph = StateGraph(TicketState)

graph.add_node("normalize", normalize)
graph.add_node("classify", classify)

graph.add_edge(START, "normalize")
graph.add_edge("normalize", "classify")
graph.add_edge("classify", END)

compiled_graph = graph.compile()

In [16]:
result = compiled_graph.invoke({
    "query": "I am not able to login to my account"
})

In [17]:
print(result)

{'query': 'I am not able to login to my account', 'normalized': 'i am not able to login to my account', 'classification': 'account'}
